In [1]:
import numpy as np
import pandas as pd

In [2]:
def calculate_angle_3d_2d(a, b, c):
    """
    Calculates the interior angle (in degrees) at vertex 'b' given three 2D points.
    Formula uses the vector dot product: cos(theta) = (ba . bc) / (|ba| * |bc|)
    """
    a = np.array(a)  # e.g., Shoulder
    b = np.array(b)  # e.g., Hip (Vertex)
    c = np.array(c)  # e.g., Knee

    # Create vectors pointing away from the vertex (b)
    ba = a - b
    bc = c - b

    # Calculate dot product and magnitudes
    dot_product = np.dot(ba, bc)
    magnitude_ba = np.linalg.norm(ba)
    magnitude_bc = np.linalg.norm(bc)

    # Prevent division by zero errors
    if magnitude_ba == 0 or magnitude_bc == 0:
        return 0.0

    # Calculate cosine and clip between -1 and 1 to prevent floating-point errors in arccos
    cosine_angle = dot_product / (magnitude_ba * magnitude_bc)
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)

    # Convert radians to degrees
    angle = np.degrees(np.arccos(cosine_angle))
    return angle


def extract_geometry_features_per_rep(rep_df):
    """
    Takes a dataframe containing exactly ONE repetition and calculates
    all 8 required Geometry & Joint Angle summary features.
    """
    # Locate the frame index representing the absolute bottom of the squat (max hip_y_smooth)
    bottom_idx = rep_df['hip_y_smooth'].idxmax()
    bottom_frame = rep_df.loc[bottom_idx]

    # --- 1. Extract raw coordinates at the bottom of the squat ---
    shoulder_b = [bottom_frame['shoulder_x_smooth'], bottom_frame['shoulder_y_smooth']]
    hip_b = [bottom_frame['hip_x_smooth'], bottom_frame['hip_y_smooth']]
    knee_b = [bottom_frame['knee_x_smooth'], bottom_frame['knee_y_smooth']]
    ankle_b = [bottom_frame['ankle_x_smooth'], bottom_frame['ankle_y_smooth']]

    # --- 2. Calculate Angles at the bottom ---
    min_knee_angle = calculate_angle_3d_2d(hip_b, knee_b, ankle_b)
    min_hip_angle = calculate_angle_3d_2d(shoulder_b, hip_b, knee_b)

    # --- 3. Calculate Maximum Torso Lean across the ENTIRE rep ---
    # Torso vector is hip-to-shoulder. A true vertical vector pointing down is [0, 1].
    torso_angles = []
    for _, frame in rep_df.iterrows():
        h = np.array([frame['hip_x_smooth'], frame['hip_y_smooth']])
        s = np.array([frame['shoulder_x_smooth'], frame['shoulder_y_smooth']])
        torso_vector = s - h  # Points from hip up to shoulder
        vertical_vector = np.array([0, -1])  # Straight up vector
        
        # Angle relative to vertical
        dot = np.dot(torso_vector, vertical_vector)
        mag_t = np.linalg.norm(torso_vector)
        mag_v = np.linalg.norm(vertical_vector)
        
        if mag_t > 0:
            cos_t = np.clip(dot / (mag_t * mag_v), -1.0, 1.0)
            torso_angles.append(np.degrees(np.arccos(cos_t)))
            
    max_torso_lean_degrees = max(torso_angles) if torso_angles else 0.0

    # --- 4. Upgraded Depth Ratio (Invariant to camera distance) ---
    # Leg length baseline calculated at the bottom frame
    hip_to_knee_len = np.linalg.norm(np.array(hip_b) - np.array(knee_b))
    knee_to_ankle_len = np.linalg.norm(np.array(knee_b) - np.array(ankle_b))
    total_leg_length = hip_to_knee_len + knee_to_ankle_len

    # Absolute depth = vertical distance from hip to ankle at bottom
    vertical_depth = bottom_frame['ankle_y_smooth'] - bottom_frame['hip_y_smooth']
    max_depth_ratio = vertical_depth / total_leg_length if total_leg_length > 0 else 0.0

    # --- 5. Hip-to-Ankle Alignment (Horizontal shift at the bottom) ---
    hip_to_ankle_bottom_alignment = abs(bottom_frame['hip_x_smooth'] - bottom_frame['ankle_x_smooth'])

    # --- 6 & 7. Average Angles During Whole Descent Phase ---
    # Descent phase is everything from the start of the rep up to the bottom frame
    descent_df = rep_df.loc[:bottom_idx]
    
    knee_angles_descent = []
    hip_angles_descent = []
    
    for _, frame in descent_df.iterrows():
        s = [frame['shoulder_x_smooth'], frame['shoulder_y_smooth']]
        h = [frame['hip_x_smooth'], frame['hip_y_smooth']]
        k = [frame['knee_x_smooth'], frame['knee_y_smooth']]
        a = [frame['ankle_x_smooth'], frame['ankle_y_smooth']]
        
        knee_angles_descent.append(calculate_angle_3d_2d(h, k, a))
        hip_angles_descent.append(calculate_angle_3d_2d(s, h, k))
        
    avg_knee_angle_descent = np.mean(knee_angles_descent) if knee_angles_descent else 0.0
    avg_hip_angle_descent = np.mean(hip_angles_descent) if hip_angles_descent else 0.0

    # --- 8. Max Shin Forward Angle ---
    # Angle of knee-to-ankle relative to vertical vector [0, -1]
    shin_angles = []
    for _, frame in rep_df.iterrows():
        k = np.array([frame['knee_x_smooth'], frame['knee_y_smooth']])
        a = np.array([frame['ankle_x_smooth'], frame['ankle_y_smooth']])
        shin_vector = k - a  # Points from ankle up to knee
        vertical_vector = np.array([0, -1])
        
        dot = np.dot(shin_vector, vertical_vector)
        mag_s = np.linalg.norm(shin_vector)
        
        if mag_s > 0:
            cos_s = np.clip(dot / (mag_s * 1.0), -1.0, 1.0)
            shin_angles.append(np.degrees(np.arccos(cos_s)))
            
    max_shin_forward_angle = max(shin_angles) if shin_angles else 0.0

    # Return as a clean dictionary ready to form your final vector row
    return {
        'min_knee_angle': min_knee_angle,
        'min_hip_angle': min_hip_angle,
        'max_torso_lean_degrees': max_torso_lean_degrees,
        'max_depth_ratio': max_depth_ratio,
        'hip_to_ankle_bottom_alignment': hip_to_ankle_bottom_alignment,
        'avg_knee_angle_descent': avg_knee_angle_descent,
        'avg_hip_angle_descent': avg_hip_angle_descent,
        'max_shin_forward_angle': max_shin_forward_angle
    }